In [2]:
#验证模型过拟合和欠拟合
import torch
import math
import numpy as np
from torch import nn
from d2l import torch as d2l

In [11]:
#生成三阶多项式数据集
max_degree = 20
n_train,n_test = 100,100
true_w = np.zeros(max_degree)
true_w[0:4]=np.array([5,1.2,-3.4,5.6])
features = np.random.normal(size=(n_train+n_test,1))
np.random.shuffle(features)
poly_features=np.power(features,np.arange(max_degree).reshape(1,max_degree))
for i in range(max_degree):
    poly_features[:,i]/=math.gamma(i+1)
labels = np.dot(poly_features,true_w)
labels+=np.random.normal(scale = 0.1,size = labels.shape)

In [19]:
true_w,features,poly_features,labels = [

    torch.tensor(x,dtype=torch.float32)
    for x in [true_w,features,poly_features,labels]
]
features[:2],poly_features[:2,:],labels[:2]

C:\Users\Rurora\AppData\Local\Temp\ipykernel_181832\1087541390.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(x,dtype=torch.float32)


(tensor([[-1.5751],
         [-1.3198]]),
 tensor([[ 1.0000e+00, -1.5751e+00,  1.2404e+00, -6.5127e-01,  2.5645e-01,
          -8.0787e-02,  2.1208e-02, -4.7720e-03,  9.3954e-04, -1.6443e-04,
           2.5899e-05, -3.7084e-06,  4.8676e-07, -5.8976e-08,  6.6352e-09,
          -6.9673e-10,  6.8588e-11, -6.3548e-12,  5.5608e-13, -4.6098e-14],
         [ 1.0000e+00, -1.3198e+00,  8.7096e-01, -3.8317e-01,  1.2643e-01,
          -3.3373e-02,  7.3410e-03, -1.3841e-03,  2.2835e-04, -3.3487e-05,
           4.4196e-06, -5.3028e-07,  5.8323e-08, -5.9213e-09,  5.5821e-10,
          -4.9116e-11,  4.0515e-12, -3.1455e-13,  2.3064e-14, -1.6021e-15]]),
 tensor([-4.8604, -1.3988]))

In [20]:
def evaluate_loss(net,data_iter,loss):
    metric = d2l.Accumulator(2)
    for x,y in data_iter:
        out = net(x)
        y = y.reshape(out.shape)
        l=loss(out,y)
    metric.add(l.sum(),l.numel())
    return metric[0]/metric[1]